In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments_clean.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Shape: (990, 16)


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [2]:
# Select numerical features
numerical_features = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows"
]

# Select categorical features
categorical_features = [
    "booking_channel",
    "visit_type"
]

# Convert reminder_sent into 1 and 0
df["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Create the feature matrix X
X = df[numerical_features + ["reminder_sent"] + categorical_features]

# Convert categorical columns into dummy variables
X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

# Create target variable y
y = df["no_show"]

print("Features created successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

Features created successfully!
X shape: (990, 9)
y shape: (990,)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Follow-up']


In [3]:
# Recreate X with all categories represented
X = df[numerical_features + ["reminder_sent"] + categorical_features]

X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

print("Corrected feature matrix!")
print("X shape:", X.shape)
print("\nFeature columns:")
print(X.columns.tolist())

Corrected feature matrix!
X shape: (990, 11)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training and testing split completed!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("\nNo-shows in test set:", y_test.sum())

Training and testing split completed!
X_train: (792, 11)
X_test: (198, 11)
y_train: (792,)
y_test: (198,)

No-shows in test set: 49


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create logistic regression pipeline
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predict test results
y_pred = logistic_model.predict(X_test)

# Predict probabilities of no-show
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Calculate performance metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7879
Precision: 0.6296
Recall: 0.3469
F1-score: 0.4474
ROC-AUC: 0.7889

Confusion Matrix:
[[139  10]
 [ 32  17]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.93      0.87       149
           1       0.63      0.35      0.45        49

    accuracy                           0.79       198
   macro avg       0.72      0.64      0.66       198
weighted avg       0.77      0.79      0.76       198



In [7]:
# Extract logistic regression coefficients
coefficients = logistic_model.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": coefficients,
    "odds_ratio": np.exp(coefficients)
})

coef_df = coef_df.sort_values(
    by="coefficient",
    ascending=False
)

display(coef_df)

,feature,coefficient,odds_ratio
4,prior_no_shows,0.649612,1.914797
3,lead_days,0.614471,1.848678
2,fee_inr,0.452761,1.572649
1,distance_km,0.253448,1.288460
7,booking_channel_Phone,0.215881,1.240955
9,visit_type_Consultation,0.015658,1.015781
10,visit_type_Follow-up,-0.015658,0.984464
6,booking_channel_App,-0.046804,0.954275
8,booking_channel_Walk-In,-0.186364,0.829971
0,age,-0.253880,0.775785


In [8]:
# Get the reminder_sent coefficient
reminder_coef = coefficients[list(X.columns).index("reminder_sent")]

# Get the standard deviation used by StandardScaler
reminder_scale = logistic_model.named_steps["scaler"].scale_[
    list(X.columns).index("reminder_sent")
]

# Calculate actual odds ratio for No -> Yes
actual_reminder_odds_ratio = np.exp(reminder_coef / reminder_scale)

print("Reminder coefficient:", round(reminder_coef, 4))
print("Standard deviation:", round(reminder_scale, 4))
print("Odds ratio (one standard deviation):", round(np.exp(reminder_coef), 4))
print("Odds ratio (No to Yes):", round(actual_reminder_odds_ratio, 4))

Reminder coefficient: -0.5807
Standard deviation: 0.495
Odds ratio (one standard deviation): 0.5595
Odds ratio (No to Yes): 0.3094


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Full Decision Tree
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# Depth-3 Decision Tree
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_depth3.fit(X_train, y_train)

# Calculate accuracy
print("Full Decision Tree")
print("Training accuracy:", round(
    accuracy_score(y_train, tree_full.predict(X_train)), 4
))
print("Testing accuracy:", round(
    accuracy_score(y_test, tree_full.predict(X_test)), 4
))

print("\nDepth-3 Decision Tree")
print("Training accuracy:", round(
    accuracy_score(y_train, tree_depth3.predict(X_train)), 4
))
print("Testing accuracy:", round(
    accuracy_score(y_test, tree_depth3.predict(X_test)), 4
))

Full Decision Tree
Training accuracy: 1.0
Testing accuracy: 0.6717

Depth-3 Decision Tree
Training accuracy: 0.822
Testing accuracy: 0.7778


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score
)

# Create Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("Random Forest Results")
print("Training accuracy:",
      round(accuracy_score(y_train, rf_model.predict(X_train)), 4))

print("Testing accuracy:",
      round(accuracy_score(y_test, rf_pred), 4))

print("Recall:",
      round(recall_score(y_test, rf_pred), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test, rf_prob), 4))

Random Forest Results
Training accuracy: 1.0
Testing accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057


In [11]:
# Extract feature importance
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
})

# Sort from highest to lowest
importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print("Top 3 Important Features:")
display(importance_df.head(3))

Top 3 Important Features:


,feature,importance
3,lead_days,0.219905
0,age,0.196788
1,distance_km,0.194739


In [12]:
from sklearn.model_selection import GridSearchCV

# Define parameters to test
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, None],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [1, 5]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

# Fit using training data only
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation ROC-AUC:", round(grid_search.best_score_, 4))

Best parameters: {'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 200}
Best cross-validation ROC-AUC: 0.7723


In [13]:
from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree (Depth 3)": tree_depth3,
    "Random Forest": rf_model,
    "Tuned Random Forest": grid_search.best_estimator_
}

for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=5,
        scoring="roc_auc"
    )
    print(name, ":", round(scores.mean(), 4))

Logistic Regression : 0.7737
Decision Tree (Depth 3) : 0.6784
Random Forest : 0.7466
Tuned Random Forest : 0.7723


In [14]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Generate out-of-fold probabilities using Logistic Regression
oof_prob = cross_val_predict(
    logistic_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Compare two thresholds
for threshold in [0.50, 0.35]:
    oof_pred = (oof_prob >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Precision:", round(precision_score(y_train, oof_pred), 4))
    print("Recall:", round(recall_score(y_train, oof_pred), 4))
    print("F1-score:", round(f1_score(y_train, oof_pred), 4))
    print("Appointments flagged:", oof_pred.sum())
    print("-----------------------")

Threshold: 0.5
Precision: 0.5753
Recall: 0.2485
F1-score: 0.3471
Appointments flagged: 73
-----------------------
Threshold: 0.35
Precision: 0.513
Recall: 0.4675
F1-score: 0.4892
Appointments flagged: 154
-----------------------


In [15]:
# Select Logistic Regression as the final model
final_model = logistic_model

# Train on the complete training dataset
final_model.fit(X_train, y_train)

# Predict probabilities on the test dataset
final_prob = final_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
best_threshold = 0.35
final_pred = (final_prob >= best_threshold).astype(int)

# Evaluate final model
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

print("Final Model: Logistic Regression")
print("Threshold:", best_threshold)
print("Accuracy:", round(accuracy_score(y_test, final_pred), 4))
print("Precision:", round(precision_score(y_test, final_pred), 4))
print("Recall:", round(recall_score(y_test, final_pred), 4))
print("F1-score:", round(f1_score(y_test, final_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, final_prob), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, final_pred))

Final Model: Logistic Regression
Threshold: 0.35
Accuracy: 0.7879
Precision: 0.5714
Recall: 0.5714
F1-score: 0.5714
ROC-AUC: 0.7889
Confusion Matrix:
[[128  21]
 [ 21  28]]


In [16]:
# Create the risk scores file
test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": final_prob,
    "flag": final_pred
})

# Save as CSV
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("File created successfully!")
print("Shape:", test_risk_scores.shape)
display(test_risk_scores.head())

File created successfully!
Shape: (198, 4)


,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.021565,0
1,A0336,0,0.150917,0
2,A0508,0,0.518692,1
3,A0314,0,0.191300,0
4,A0089,0,0.167718,0


In [17]:
from google.colab import files
files.download("test_risk_scores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>